# Full Fine-Tuning — RECAST Constraint-Following

Notebook version of `train_full_finetune.py`.  
Defaults to `meta-llama/Llama-3.2-1B-Instruct` trained on the local RECAST-30K dataset.

**Loss:**  
`total_loss = ce_loss + lambda * constraint_loss`  
where `constraint_loss = 1 - mean(constraint_score)` over the batch.

## 1. Imports

In [ ]:
import json
import logging
import os
import random
import sys
import zipfile
from pathlib import Path

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)
from datasets import Dataset

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger(__name__)

## 2. Configuration

Edit the values below to change model, dataset, output directory, and hyperparameters.

In [ ]:
SCRIPT_DIR = Path(".").resolve()
DEFAULT_DATASET = str(SCRIPT_DIR / "datasets" / "RECAST-30K.json")
DEFAULT_MODEL   = "meta-llama/Llama-3.2-1B-Instruct"

# ── Hyperparameters ──────────────────────────────────────────────────────────
MODEL_NAME                   = DEFAULT_MODEL
DATASET_PATH                 = DEFAULT_DATASET
OUTPUT_DIR                   = "./output/finetuned"
NUM_TRAIN_EPOCHS             = 2
PER_DEVICE_TRAIN_BATCH_SIZE  = 4
PER_DEVICE_EVAL_BATCH_SIZE   = 4
GRADIENT_ACCUMULATION_STEPS  = 8
LEARNING_RATE                = 2e-5
LR_SCHEDULER_TYPE            = "cosine"
WARMUP_RATIO                 = 0.03
MAX_LENGTH                   = 512
VAL_SPLIT                    = 0.1     # 90% train / 10% validation
SEED                         = 42
LOGGING_STEPS                = 50
SAVE_STEPS                   = 500
EVAL_STEPS                   = 500
GRADIENT_CHECKPOINTING       = True
NUM_SAMPLES                  = 0       # 0 = use all data
CONSTRAINT_LAMBDA            = 0.1    # set to 0 to use standard CE loss only
HF_TOKEN                     = os.environ.get("HF_TOKEN", "")

## 3. Rule-Based Constraint Helpers

In [ ]:
def _parse_length_constraint(rule_dict: dict):
    wl = rule_dict.get("word_length")
    if not wl:
        return None
    fi = wl.get("func_input", [])
    if len(fi) < 2:
        return None
    range_ = fi[1]
    if isinstance(range_, list) and len(range_) == 2:
        return (int(range_[0]), int(range_[1]))
    target = fi[2] if len(fi) > 2 and fi[2] is not None else None
    if target is not None:
        return (max(1, int(target * 0.8)), int(target * 1.2))
    return None


def _parse_keyword_constraint(rule_dict: dict):
    kw = rule_dict.get("keyword")
    if not kw:
        return None
    fi = kw.get("func_input", [])
    return fi[1] if len(fi) > 1 and isinstance(fi[1], dict) else None


def _parse_start_with_constraint(rule_dict: dict):
    sw = rule_dict.get("start_with")
    if not sw:
        return None
    fi = sw.get("func_input", [])
    return fi[1] if len(fi) > 1 and isinstance(fi[1], str) else None


def _parse_end_with_constraint(rule_dict: dict):
    ew = rule_dict.get("end_with")
    if not ew:
        return None
    fi = ew.get("func_input", [])
    return fi[1] if len(fi) > 1 and isinstance(fi[1], str) else None


def _constraint_score(response: str, rule_dict: dict) -> float:
    total = 0
    passed = 0

    lp = _parse_length_constraint(rule_dict)
    if lp:
        total += 1
        wc = len(response.split())
        passed += 1 if lp[0] <= wc <= lp[1] else 0

    kw = _parse_keyword_constraint(rule_dict)
    if kw:
        total += 1
        resp_lower = response.lower()
        ok = all(resp_lower.count(k.lower()) >= int(v) for k, v in kw.items())
        passed += 1 if ok else 0

    sw = _parse_start_with_constraint(rule_dict)
    if sw:
        total += 1
        passed += 1 if response.strip().lower().startswith(sw.strip().lower()) else 0

    ew = _parse_end_with_constraint(rule_dict)
    if ew:
        total += 1
        passed += 1 if response.strip().lower().endswith(ew.strip().lower()) else 0

    return passed / total if total > 0 else 1.0

## 4. Dataset Helpers

In [ ]:
def _infer_difficulty(num_constraints: int) -> str:
    if num_constraints <= 2:
        return "L1"
    elif num_constraints <= 4:
        return "L2"
    elif num_constraints <= 7:
        return "L3"
    return "L4"


def _parse_record(record: dict, idx: int) -> dict:
    prompt = (
        record.get("winner_prompt")
        or record.get("prompt")
        or record.get("instruction")
        or ""
    )
    response = (
        record.get("winner_response")
        or record.get("response")
        or record.get("output")
        or ""
    )
    constraints = record.get("constraints", record.get("constraint_list")) or []
    if isinstance(constraints, str):
        try:
            constraints = json.loads(constraints)
        except json.JSONDecodeError:
            constraints = [{"type": "raw", "description": constraints}]
    if not isinstance(constraints, list):
        constraints = [constraints] if constraints else []

    difficulty = (
        record.get("difficulty_level")
        or record.get("difficulty")
        or record.get("level")
        or _infer_difficulty(len(constraints))
    )
    difficulty = str(difficulty)
    if not difficulty.startswith("L"):
        difficulty = f"L{difficulty}"

    return {
        "id": str(record.get("id", idx)),
        "prompt": prompt,
        "response": response,
        "difficulty_level": difficulty,
        "rule_evaluate_dict": record.get("rule_evaluate_dict", {}),
    }


def _resolve_dataset_path(dataset_arg: str) -> str:
    p = Path(dataset_arg)
    if p.suffix != ".zip" or not p.exists():
        return dataset_arg
    extract_dir = p.parent
    logger.info(f"Extracting {p} → {extract_dir}")
    with zipfile.ZipFile(p, "r") as z:
        z.extractall(extract_dir)
    candidates = sorted(
        f for f in extract_dir.iterdir()
        if f.suffix in (".jsonl", ".json") and f.stem != p.stem
    )
    if not candidates:
        raise RuntimeError(f"No .jsonl/.json file found after extracting {p}")
    logger.info(f"Using extracted file: {candidates[0]}")
    return str(candidates[0])


def load_recast_dataset(dataset_path: str) -> list[dict]:
    dataset_path = _resolve_dataset_path(dataset_path)
    path = Path(dataset_path)

    if path.exists():
        if path.stat().st_size < 1024:
            first_bytes = path.read_bytes()
            if b"git-lfs" in first_bytes:
                raise RuntimeError(
                    f"{path} is a Git LFS pointer. "
                    "Run `git lfs pull` to download the actual dataset file before training."
                )
        logger.info(f"Loading dataset from local file: {path}")
        raw = []
        if path.suffix == ".jsonl":
            with open(path) as f:
                for line in f:
                    line = line.strip()
                    if line:
                        raw.append(json.loads(line))
        else:
            with open(path) as f:
                loaded = json.load(f)
            if isinstance(loaded, list):
                raw = loaded
            elif isinstance(loaded, dict):
                for key in ("data", "instances", "examples", "records"):
                    if key in loaded and isinstance(loaded[key], list):
                        raw = loaded[key]
                        break
                if not raw:
                    raw = [loaded]
    else:
        logger.info(f"Loading dataset from HuggingFace Hub: {dataset_path}")
        from datasets import load_dataset as hf_load
        hf_ds = hf_load(dataset_path, split="train")
        raw = [dict(row) for row in hf_ds]

    records = [_parse_record(r, i) for i, r in enumerate(raw)]
    logger.info(f"Loaded {len(records)} records")
    return records

## 5. Tokenisation

In [ ]:
def build_tokenised_dataset(
    records: list[dict],
    tokenizer: AutoTokenizer,
    max_length: int,
) -> Dataset:
    has_template = (
        hasattr(tokenizer, "chat_template") and tokenizer.chat_template is not None
    )

    def tokenise(record):
        prompt_text   = record["prompt"]
        response_text = record["response"]
        rule_dict     = record.get("rule_evaluate_dict") or {}

        if has_template:
            full_chat = [
                {"role": "user",      "content": prompt_text},
                {"role": "assistant", "content": response_text},
            ]
            prompt_chat = [{"role": "user", "content": prompt_text}]
            full_text             = tokenizer.apply_chat_template(full_chat,   tokenize=False, add_generation_prompt=False)
            prompt_text_formatted = tokenizer.apply_chat_template(prompt_chat, tokenize=False, add_generation_prompt=True)
        else:
            prompt_text_formatted = f"User: {prompt_text}\nAssistant:"
            full_text             = f"{prompt_text_formatted} {response_text}"

        full_enc   = tokenizer(full_text,             max_length=max_length, truncation=True, padding=False)
        prompt_enc = tokenizer(prompt_text_formatted, max_length=max_length, truncation=True, padding=False)

        input_ids  = full_enc["input_ids"]
        labels     = list(input_ids)
        prompt_len = len(prompt_enc["input_ids"])
        labels[:prompt_len] = [-100] * prompt_len

        if all(lbl == -100 for lbl in labels):
            return {"input_ids": None, "attention_mask": None, "labels": None, "constraint_score": None}

        return {
            "input_ids":        input_ids,
            "attention_mask":   full_enc["attention_mask"],
            "labels":           labels,
            "constraint_score": _constraint_score(response_text, rule_dict),
        }

    logger.info("Tokenising dataset …")
    hf_ds     = Dataset.from_list(records)
    tokenised = hf_ds.map(tokenise, remove_columns=hf_ds.column_names, desc="Tokenising", num_proc=1)
    before    = len(tokenised)
    tokenised = tokenised.filter(lambda x: x["input_ids"] is not None)
    dropped   = before - len(tokenised)
    if dropped:
        logger.warning(f"Dropped {dropped} examples where prompt exceeded max_length={max_length}")
    return tokenised

## 6. Constraint-Aware Training Components

In [ ]:
class ConstraintDataCollator(DataCollatorForSeq2Seq):
    """Extends DataCollatorForSeq2Seq to pass constraint_score through to the batch."""

    def __call__(self, features):
        scores = [f.pop("constraint_score", 1.0) for f in features]
        batch  = super().__call__(features)
        batch["constraint_score"] = torch.tensor(scores, dtype=torch.float32)
        return batch


class ConstraintAwareTrainer(Trainer):
    """
    total_loss = ce_loss + lambda * constraint_loss
    constraint_loss = 1 - mean(constraint_score) over the batch.
    """

    def __init__(self, *args, constraint_lambda: float = 0.1, **kwargs):
        super().__init__(*args, **kwargs)
        self.constraint_lambda = constraint_lambda

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        constraint_scores = inputs.pop("constraint_score", None)
        outputs   = model(**inputs)
        ce_loss   = outputs.loss

        if constraint_scores is not None and self.constraint_lambda > 0:
            constraint_loss = 1.0 - constraint_scores.float().mean()
            loss = ce_loss + self.constraint_lambda * constraint_loss
        else:
            constraint_loss = torch.tensor(0.0)
            loss = ce_loss

        if self.state.global_step % self.args.logging_steps == 0:
            self.log({
                "ce_loss":         ce_loss.detach().item(),
                "constraint_loss": constraint_loss.detach().item(),
            })

        return (loss, outputs) if return_outputs else loss

## 7. Load & Split Data

In [ ]:
random.seed(SEED)

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    logger.info("Logged in to HuggingFace Hub.")

records = load_recast_dataset(DATASET_PATH)

if NUM_SAMPLES > 0:
    random.shuffle(records)
    records = records[:NUM_SAMPLES]
    logger.info(f"Subsampled to {len(records)} records")

random.shuffle(records)
n_val         = max(1, int(len(records) * VAL_SPLIT))
val_records   = records[:n_val]
train_records = records[n_val:]
logger.info(f"Train: {len(train_records)}  Val: {len(val_records)}")

## 8. Load Tokenizer & Tokenise

In [ ]:
logger.info(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

train_ds = build_tokenised_dataset(train_records, tokenizer, MAX_LENGTH)
val_ds   = build_tokenised_dataset(val_records,   tokenizer, MAX_LENGTH)

## 9. Load Model

In [ ]:
logger.info(f"Loading model: {MODEL_NAME}")
n_gpus     = torch.cuda.device_count()
device_map = "auto" if n_gpus <= 1 else None
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map=device_map,
)

if GRADIENT_CHECKPOINTING:
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    logger.info("Gradient checkpointing enabled.")

for param in model.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
logger.info(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

## 10. Training Arguments

In [ ]:
safe_model_name = MODEL_NAME.replace("/", "_")
training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = NUM_TRAIN_EPOCHS,
    per_device_train_batch_size = PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size  = PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps = GRADIENT_ACCUMULATION_STEPS,
    learning_rate               = LEARNING_RATE,
    lr_scheduler_type           = LR_SCHEDULER_TYPE,
    warmup_ratio                = WARMUP_RATIO,
    bf16                        = torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16                        = False,
    logging_dir                 = os.path.join(OUTPUT_DIR, "logs"),
    logging_steps               = LOGGING_STEPS,
    eval_strategy               = "steps",
    eval_steps                  = EVAL_STEPS,
    save_strategy               = "steps",
    save_steps                  = SAVE_STEPS,
    save_total_limit            = 3,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    report_to                   = "none",
    seed                        = SEED,
    dataloader_num_workers      = 0,
    remove_unused_columns       = False,
    label_names                 = ["labels"],
    run_name                    = f"full_ft_{safe_model_name}",
)

## 11. Build Trainer & Train

In [ ]:
use_constraint_loss = CONSTRAINT_LAMBDA > 0

if use_constraint_loss:
    logger.info(
        f"Constraint-aware loss enabled (lambda={CONSTRAINT_LAMBDA}). "
        "total_loss = ce_loss + lambda * constraint_loss"
    )
    data_collator = ConstraintDataCollator(
        tokenizer=tokenizer, padding=True, pad_to_multiple_of=8, label_pad_token_id=-100
    )
    trainer = ConstraintAwareTrainer(
        model             = model,
        args              = training_args,
        train_dataset     = train_ds,
        eval_dataset      = val_ds,
        data_collator     = data_collator,
        processing_class  = tokenizer,
        constraint_lambda = CONSTRAINT_LAMBDA,
    )
else:
    logger.info("Constraint loss disabled. Using standard CE loss.")
    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer, padding=True, pad_to_multiple_of=8, label_pad_token_id=-100
    )
    trainer = Trainer(
        model            = model,
        args             = training_args,
        train_dataset    = train_ds,
        eval_dataset     = val_ds,
        data_collator    = data_collator,
        processing_class = tokenizer,
    )

logger.info("Starting full fine-tuning …")
trainer.train()

## 12. Save Model & Metadata

In [ ]:
logger.info(f"Saving model to {OUTPUT_DIR}")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

meta = {
    "model_name":           MODEL_NAME,
    "dataset":              DATASET_PATH,
    "num_train_epochs":     NUM_TRAIN_EPOCHS,
    "effective_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
    "learning_rate":        LEARNING_RATE,
    "max_length":           MAX_LENGTH,
    "train_samples":        len(train_records),
    "val_samples":          len(val_records),
    "constraint_lambda":    CONSTRAINT_LAMBDA,
}
with open(os.path.join(OUTPUT_DIR, "train_meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

logger.info("Done.")
print(meta)